In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Fri Oct 30 12:27:53 2020

@author: ifenty
"""
import os
import sys
import time
import hashlib
import json
import numpy as np
import importlib
sys.path.append(os.path.expanduser('~/git_repos/ECCOv4-py'))
import ecco_v4_py as ecco

# make changes to example.py file

importlib.reload(ecco)
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import pandas as pd
import netCDF4 as nc4
import xarray as xr
import datetime
from pprint import pprint
from collections import OrderedDict
import pyresample as pr
import uuid
import pickle
import matplotlib.pyplot as plt
from pandas import read_csv

In [3]:
tmp_sassie_grid = xr.open_dataset('/Users/ifenty/Projects/SASSIE/V1r1/SASSIE_ECCO_L4_GEOMETRY_LLC1080GRID_V1R1/GRID_GEOMETRY_SASSIE_ECCO_V1R1_native_llc1080.nc')
tmp_sassie_grid.close()
tmp_sassie_grid

<xarray.Dataset> Size: 4GB
Dimensions:     (j: 1080, j_g: 1080, i: 1800, i_g: 1800, k: 90, k_u: 90,
                 k_l: 90, k_p1: 91, nb: 4, nv: 2)
Coordinates: (12/23)
  * j           (j) int32 4kB 0 1 2 3 4 5 6 ... 1074 1075 1076 1077 1078 1079
  * j_g         (j_g) int32 4kB 0 1 2 3 4 5 6 ... 1074 1075 1076 1077 1078 1079
  * i           (i) int32 7kB 0 1 2 3 4 5 6 ... 1794 1795 1796 1797 1798 1799
  * i_g         (i_g) int32 7kB 0 1 2 3 4 5 6 ... 1794 1795 1796 1797 1798 1799
  * k           (k) int32 360B 0 1 2 3 4 5 6 7 8 ... 81 82 83 84 85 86 87 88 89
  * k_u         (k_u) int32 360B 0 1 2 3 4 5 6 7 8 ... 82 83 84 85 86 87 88 89
    ...          ...
    Zp1         (k_p1) float32 364B ...
    Zu          (k_u) float32 360B ...
    Zl          (k_l) float32 360B ...
    XC_bnds     (j, i, nb) float32 31MB ...
    YC_bnds     (j, i, nb) float32 31MB ...
    Z_bnds      (k, nv) float32 720B ...
Dimensions without coordinates: nb, nv
Data variables: (12/22)
    CS          (j, i) float32 8MB ...
    SN          (j, i) float32 8MB ...
    rAc         (j, i) float32 8MB ...
    dxG         (j_g, i) float32 8MB ...
    dyG         (j, i_g) float32 8MB ...
    Depth       (j, i) float32 8MB ...
    ...          ...
    hFacW       (k, j, i_g) float32 700MB ...
    hFacS       (k, j_g, i) float32 700MB ...
    maskC       (k, j, i) int8 175MB ...
    maskW       (k, j, i_g) int8 175MB ...
    maskS       (k, j_g, i) int8 175MB ...
    mask_basin  (k, j, i) int32 700MB ...
Attributes: (12/58)
    acknowledgement:                   This research was carried out by the J...
    author:                            Marie Zahn, Mike Wood, Ian Fenty, and ...
    cdm_data_type:                     Grid
    comment:                           Fields provided on the curvilinear lat...
    Conventions:                       CF-1.8, ACDD-1.3
    creator_email:                     marie.j.zahn@jpl.nasa.gov
    ...                                ...
    standard_name_vocabulary:          NetCDF Climate and Forecast (CF) Metad...
    summary:                           This dataset provides geometric parame...
    title:                             SASSIE ECCO Geometry Parameters - llc1...
    uuid:                              5f721686-dd26-11f0-8c8d-0604868e061f
    identifier_product_doi:            https://doi.org/10.5067/SEL1S-GEO11
    identifier_product_doi_authority:  org.doi

In [12]:
tmp_v4r6_grid = xr.open_dataset('/Users/ifenty/tmp/GRID_GEOMETRY_ECCO_V4r6_native_llc0090.nc');
tmp_v4r6_grid.close()
tmp_v4r6_grid

<xarray.Dataset> Size: 208MB
Dimensions:                 (i: 90, i_g: 90, j: 90, j_g: 90, k: 50, k_u: 50,
                             k_l: 50, k_p1: 51, tile: 13, nb: 4, nv: 2,
                             j_gg: 91, i_gg: 91)
Coordinates: (12/26)
  * i                       (i) int32 360B 0 1 2 3 4 5 6 ... 84 85 86 87 88 89
  * i_g                     (i_g) int32 360B 0 1 2 3 4 5 6 ... 84 85 86 87 88 89
  * j                       (j) int32 360B 0 1 2 3 4 5 6 ... 84 85 86 87 88 89
  * j_g                     (j_g) int32 360B 0 1 2 3 4 5 6 ... 84 85 86 87 88 89
  * k                       (k) int32 200B 0 1 2 3 4 5 6 ... 44 45 46 47 48 49
  * k_u                     (k_u) int32 200B 0 1 2 3 4 5 6 ... 44 45 46 47 48 49
    ...                      ...
    XGG                     (tile, j_gg, i_gg) float32 431kB ...
    YGG                     (tile, j_gg, i_gg) float32 431kB ...
    XV                      (tile, j_g, i) float32 421kB ...
    YV                      (tile, j_g, i) float32 421kB ...
    XU                      (tile, j, i_g) float32 421kB ...
    YU                      (tile, j, i_g) float32 421kB ...
Dimensions without coordinates: nb, nv, j_gg, i_gg
Data variables: (12/38)
    CS                      (tile, j, i) float32 421kB ...
    SN                      (tile, j, i) float32 421kB ...
    rA                      (tile, j, i) float32 421kB ...
    dxG                     (tile, j_g, i) float32 421kB ...
    dyG                     (tile, j, i_g) float32 421kB ...
    Depth                   (tile, j, i) float32 421kB ...
    ...                      ...
    mask3dSHI               (k, tile, j, i) float32 21MB ...
    mask3dICF               (k, tile, j, i) float32 21MB ...
    mask3dSHIICF            (k, tile, j, i) float32 21MB ...
    ocean_column_thickness  (tile, j, i) float32 421kB ...
    ocean_ice_id            (tile, j, i) float32 421kB ...
    ice_shelf_draft         (tile, j, i) float32 421kB ...
Attributes: (12/61)
    acknowledgement:                   This research was carried out by the J...
    author:                            Ian Fenty, Ou Wang, Ichiro Fukumori
    cdm_data_type:                     Grid
    comment:                           Fields provided on the curvilinear lat...
    Conventions:                       CF-1.8, ACDD-1.3
    coordinates_comment:               Note: the global 'coordinates' attribu...
    ...                                ...
    references:                        Fenty, I., Fukumori, I., Wang, O., For...
    source:                            The ECCO V4r6 state estimate was produ...
    standard_name_vocabulary:          NetCDF Climate and Forecast (CF) Metad...
    summary:                           This dataset provides geometric parame...
    title:                             ECCO Geometry Parameters - llc90 Grid ...
    uuid:                              c1057168-5887-11f1-8b79-8af43ebf959f

In [13]:
tmp_v4r4_grid_ll = xr.open_dataset('/Users/ifenty/Library/CloudStorage/Box-Box/ifenty/Projects/ECCO/ECCOv4/R4/PODAAC/grid_ECCOV4r4_20210309b/GRID_GEOMETRY_ECCO_v4r4_latlon_0p50deg.nc').load()
tmp_v4r4_grid_ll.close()
tmp_v4r4_grid_ll

<xarray.Dataset> Size: 121MB
Dimensions:         (Z: 50, latitude: 360, longitude: 720, nv: 2)
Coordinates:
  * Z               (Z) float32 200B -5.0 -15.0 -25.0 ... -5.461e+03 -5.906e+03
  * latitude        (latitude) float32 1kB -89.75 -89.25 -88.75 ... 89.25 89.75
  * longitude       (longitude) float32 3kB -179.8 -179.2 -178.8 ... 179.2 179.8
    latitude_bnds   (latitude, nv) float32 3kB -90.0 -89.5 -89.5 ... 89.5 90.0
    longitude_bnds  (longitude, nv) float32 6kB -180.0 -179.5 ... 179.5 180.0
    Z_bnds          (Z, nv) float32 400B 0.0 -10.0 ... -5.678e+03 -6.134e+03
Dimensions without coordinates: nv
Data variables:
    hFacC           (Z, latitude, longitude) float64 104MB nan nan ... nan nan
    Depth           (latitude, longitude) float64 2MB nan nan ... 4.178e+03
    area            (latitude, longitude) float64 2MB 1.349e+07 ... 1.349e+07
    drF             (Z) float32 200B 10.0 10.0 10.0 10.0 ... 410.5 433.5 456.5
    maskC           (Z, latitude, longitude) bool 13MB False False ... False
Attributes: (12/57)
    acknowledgement:                 This research was carried out by the Jet...
    author:                          Ian Fenty and Ou Wang
    cdm_data_type:                   Grid
    comment:                         Fields provided on a regular lat-lon gri...
    Conventions:                     CF-1.8, ACDD-1.3
    creator_email:                   ecco-group@mit.edu
    ...                              ...
    source:                          The ECCO V4r4 state estimate was produce...
    standard_name_vocabulary:        NetCDF Climate and Forecast (CF) Metadat...
    summary:                         This dataset provides geometric paramete...
    title:                           ECCO Geometry Parameters for the 0.5 deg...
    uuid:                            02c35eec-813f-11eb-aa29-f8f21e2ee3e0
    coordinates_comment:             Note: the global 'coordinates' attribute...

In [14]:
tmp_v4r4_grid = xr.open_dataset('/Users/ifenty/Library/CloudStorage/Box-Box/ifenty/Projects/ECCO/ECCOv4/R4/PODAAC/grid_ECCOV4r4_20210309b/GRID_GEOMETRY_ECCO_v4r4_native_llc0090.nc').load()
tmp_v4r4_grid.close()
tmp_v4r4_grid

<xarray.Dataset> Size: 89MB
Dimensions:  (i: 90, i_g: 90, j: 90, j_g: 90, k: 50, k_u: 50, k_l: 50,
              k_p1: 51, tile: 13, nb: 4, nv: 2)
Coordinates: (12/20)
  * i        (i) int32 360B 0 1 2 3 4 5 6 7 8 9 ... 81 82 83 84 85 86 87 88 89
  * i_g      (i_g) int32 360B 0 1 2 3 4 5 6 7 8 9 ... 81 82 83 84 85 86 87 88 89
  * j        (j) int32 360B 0 1 2 3 4 5 6 7 8 9 ... 81 82 83 84 85 86 87 88 89
  * j_g      (j_g) int32 360B 0 1 2 3 4 5 6 7 8 9 ... 81 82 83 84 85 86 87 88 89
  * k        (k) int32 200B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49
  * k_u      (k_u) int32 200B 0 1 2 3 4 5 6 7 8 9 ... 41 42 43 44 45 46 47 48 49
    ...       ...
    Zp1      (k_p1) float32 204B 0.0 -10.0 -20.0 ... -5.678e+03 -6.134e+03
    Zu       (k_u) float32 200B -10.0 -20.0 -30.0 ... -5.678e+03 -6.134e+03
    Zl       (k_l) float32 200B 0.0 -10.0 -20.0 ... -5.244e+03 -5.678e+03
    XC_bnds  (tile, j, i, nb) float32 2MB -115.0 -115.0 -107.9 ... -115.0 -108.5
    YC_bnds  (tile, j, i, nb) float32 2MB -88.18 -88.32 -88.3 ... -88.18 -88.16
    Z_bnds   (k, nv) float32 400B 0.0 -10.0 -10.0 ... -5.678e+03 -6.134e+03
Dimensions without coordinates: nb, nv
Data variables: (12/21)
    CS       (tile, j, i) float32 421kB 0.06158 0.06675 ... -0.9854 -0.9984
    SN       (tile, j, i) float32 421kB -0.9981 -0.9978 ... -0.1705 -0.05718
    rA       (tile, j, i) float32 421kB 3.623e+08 3.633e+08 ... 3.611e+08
    dxG      (tile, j_g, i) float32 421kB 1.558e+04 1.559e+04 ... 2.314e+04
    dyG      (tile, j, i_g) float32 421kB 2.321e+04 2.327e+04 ... 1.558e+04
    Depth    (tile, j, i) float32 421kB 0.0 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    ...       ...
    hFacC    (k, tile, j, i) float32 21MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    hFacW    (k, tile, j, i_g) float32 21MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    hFacS    (k, tile, j_g, i) float32 21MB 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0 0.0
    maskC    (k, tile, j, i) bool 5MB False False False ... False False False
    maskW    (k, tile, j, i_g) bool 5MB False False False ... False False False
    maskS    (k, tile, j_g, i) bool 5MB False False False ... False False False
Attributes: (12/60)
    acknowledgement:                 This research was carried out by the Jet...
    author:                          Ian Fenty and Ou Wang
    cdm_data_type:                   Grid
    comment:                         Fields provided on the curvilinear lat-l...
    Conventions:                     CF-1.8, ACDD-1.3
    creator_email:                   ecco-group@mit.edu
    ...                              ...
    source:                          The ECCO V4r4 state estimate was produce...
    standard_name_vocabulary:        NetCDF Climate and Forecast (CF) Metadat...
    summary:                         This dataset provides geometric paramete...
    title:                           ECCO Geometry Parameters for the Lat-Lon...
    uuid:                            d7cdfa3a-813e-11eb-aa29-f8f21e2ee3e0
    coordinates_comment:             Note: the global 'coordinates' attribute...

In [15]:
pprint(list(tmp_v4r6_grid.attrs.keys()))

['acknowledgement',
 'author',
 'cdm_data_type',
 'comment',
 'Conventions',
 'coordinates_comment',
 'creator_email',
 'creator_institution',
 'creator_name',
 'creator_type',
 'creator_url',
 'date_created',
 'date_issued',
 'date_metadata_modified',
 'date_modified',
 'geospatial_bounds_crs',
 'geospatial_lat_max',
 'geospatial_lat_min',
 'geospatial_lat_resolution',
 'geospatial_lat_units',
 'geospatial_lon_max',
 'geospatial_lon_min',
 'geospatial_lon_resolution',
 'geospatial_lon_units',
 'geospatial_vertical_max',
 'geospatial_vertical_min',
 'geospatial_vertical_positive',
 'geospatial_vertical_resolution',
 'geospatial_vertical_units',
 'history',
 'id',
 'identifier_product_doi',
 'identifier_product_doi_authority',
 'institution',
 'instrument_vocabulary',
 'keywords',
 'keywords_vocabulary',
 'license',
 'metadata_link',
 'naming_authority',
 'platform',
 'platform_vocabulary',
 'processing_level',
 'product_name',
 'product_time_coverage_end',
 'product_time_coverage_start

In [16]:
dict_A = tmp_v4r6_grid.attrs
dict_B = tmp_sassie_grid.attrs


# 1. Keys in both A and B (Intersection)
keys_in_both = dict_A.keys() & dict_B.keys()

# 2. Keys in A but not in B (Difference)
keys_in_A_not_B = dict_A.keys() - dict_B.keys()

# 3. Keys in B but not in A (Difference)
keys_in_B_not_A = dict_B.keys() - dict_A.keys()

# Print the results
print("Keys in both A and B :")
pprint(keys_in_both)

print("\nKeys in A but not B  :")
pprint(keys_in_A_not_B)
print("\nKeys in B but not A  :")
pprint(keys_in_B_not_A)

Keys in both A and B :
{'Conventions',
 'acknowledgement',
 'author',
 'cdm_data_type',
 'comment',
 'creator_email',
 'creator_institution',
 'creator_name',
 'creator_type',
 'creator_url',
 'date_created',
 'date_issued',
 'date_metadata_modified',
 'date_modified',
 'geospatial_bounds_crs',
 'geospatial_lat_max',
 'geospatial_lat_min',
 'geospatial_lat_resolution',
 'geospatial_lat_units',
 'geospatial_lon_max',
 'geospatial_lon_min',
 'geospatial_lon_resolution',
 'geospatial_lon_units',
 'geospatial_vertical_max',
 'geospatial_vertical_min',
 'geospatial_vertical_positive',
 'geospatial_vertical_resolution',
 'geospatial_vertical_units',
 'history',
 'id',
 'identifier_product_doi',
 'identifier_product_doi_authority',
 'institution',
 'instrument_vocabulary',
 'keywords',
 'keywords_vocabulary',
 'license',
 'metadata_link',
 'naming_authority',
 'platform',
 'platform_vocabulary',
 'processing_level',
 'product_name',
 'product_time_coverage_end',
 'product_time_coverage_start'

In [17]:
for attr in keys_in_both:
    print(f'\n=== {attr} ===')
    print(f'r6:  {tmp_v4r6_grid.attrs[attr]}')
    print(f'sa:  {tmp_sassie_grid.attrs[attr]}')



=== creator_institution ===
r6:  NASA Jet Propulsion Laboratory (JPL)
sa:  NASA Jet Propulsion Laboratory (JPL)

=== geospatial_lat_units ===
r6:  degrees_north
sa:  degrees_north

=== date_issued ===
r6:  2026-05-25T15:19:10
sa:  2025-12-19T22:02:12

=== platform_vocabulary ===
r6:  GCMD platform keywords
sa:  GCMD platform keywords

=== geospatial_vertical_min ===
r6:  -6134.5
sa:  -7000.0

=== institution ===
r6:  NASA Jet Propulsion Laboratory (JPL)
sa:  NASA Jet Propulsion Laboratory (JPL)

=== geospatial_lat_max ===
r6:  90.0
sa:  90.0

=== title ===
r6:  ECCO Geometry Parameters - llc90 Grid (Version 4 Release 6)
sa:  SASSIE ECCO Geometry Parameters - llc1080 Grid (Version 1 Release 1)

=== geospatial_vertical_max ===
r6:  0.0
sa:  0.0

=== acknowledgement ===
r6:  This research was carried out by the Jet Propulsion Laboratory, managed by the California Institute of Technology under a contract with the National Aeronautics and Space Administration.
sa:  This research was carrie

In [10]:
dict_A = tmp_v4r6_grid.attrs
dict_B = tmp_v4r4_grid.attrs


# 1. Keys in both A and B (Intersection)
keys_in_both = dict_A.keys() & dict_B.keys()

# 2. Keys in A but not in B (Difference)
keys_in_A_not_B = dict_A.keys() - dict_B.keys()

# 3. Keys in B but not in A (Difference)
keys_in_B_not_A = dict_B.keys() - dict_A.keys()

# Print the results
print("Keys in both A and B :")
pprint(keys_in_both)

print("\nKeys in A but not B  :")
pprint(keys_in_A_not_B)
print("\nKeys in B but not A  :")
pprint(keys_in_B_not_A)

Keys in both A and B :
{'Conventions',
 'acknowledgement',
 'author',
 'cdm_data_type',
 'comment',
 'coordinates_comment',
 'creator_email',
 'creator_institution',
 'creator_name',
 'creator_type',
 'creator_url',
 'date_created',
 'date_issued',
 'date_metadata_modified',
 'date_modified',
 'geospatial_bounds_crs',
 'geospatial_lat_max',
 'geospatial_lat_min',
 'geospatial_lat_resolution',
 'geospatial_lat_units',
 'geospatial_lon_max',
 'geospatial_lon_min',
 'geospatial_lon_resolution',
 'geospatial_lon_units',
 'geospatial_vertical_max',
 'geospatial_vertical_min',
 'geospatial_vertical_positive',
 'geospatial_vertical_resolution',
 'geospatial_vertical_units',
 'history',
 'id',
 'institution',
 'instrument_vocabulary',
 'keywords',
 'keywords_vocabulary',
 'license',
 'metadata_link',
 'naming_authority',
 'platform',
 'platform_vocabulary',
 'processing_level',
 'product_name',
 'product_time_coverage_end',
 'product_time_coverage_start',
 'product_version',
 'program',
 'proj

In [11]:
for attr in keys_in_both:
    print(f'\n=== {attr} ===')
    print(f'r6:  {tmp_v4r6_grid.attrs[attr]}')
    print(f'r4:  {tmp_v4r4_grid.attrs[attr]}')



=== creator_institution ===
r6:  NASA Jet Propulsion Laboratory (JPL)
r4:  NASA Jet Propulsion Laboratory (JPL)

=== geospatial_lat_units ===
r6:  degrees_north
r4:  degrees_north

=== date_issued ===
r6:  2026-05-25T15:15:19
r4:  2021-03-09T17:20:47

=== platform_vocabulary ===
r6:  GCMD platform keywords
r4:  GCMD platform keywords

=== geospatial_vertical_min ===
r6:  -6134.5
r4:  -6134.5

=== institution ===
r6:  NASA Jet Propulsion Laboratory (JPL)
r4:  NASA Jet Propulsion Laboratory (JPL)

=== geospatial_lat_max ===
r6:  90.0
r4:  90.0

=== title ===
r6:  ECCO Geometry Parameters - llc90 Grid (Version 4 Release 6)
r4:  ECCO Geometry Parameters for the Lat-Lon-Cap 90 (llc90) Native Model Grid (Version 4 Release 4)

=== geospatial_vertical_max ===
r6:  0.0
r4:  0.0

=== acknowledgement ===
r6:  This research was carried out by the Jet Propulsion Laboratory, managed by the California Institute of Technology under a contract with the National Aeronautics and Space Administration.
r4